In [2]:
from experiments import * 

In [6]:
def multiple_exp_jupyter (variable_parameters) : 


    #                         'attack_name': ['ALIE','FOE','Mimic','SF','PoisonedFL','MinMax','MinSum','NNP'],
    #                         'aggregator_name': ['CWMed','CwTM','RFA','Krum','Mean'],
    #                         'pre_aggregator_name': ['None','NNM','BKT','FoundFL'],
    #                         'criterion_name': ['CrossEntropy','FedLC','DMFL'],
    #                         'dataset_name': ['Purchase100', 'MNIST', 'CIFAR10', 'Fashion_MNIST'],

    gpu_list = []
    gpu_selection = 0
    

    constant_parameters = {
                    'n_workers': 5,
                    'batch_size': 8,
                    'reg_param':1e-3,
                    'clip_param': 5,
                    'beta': 0.9,
                    'seed': 1,
                    'experiment_folder':'experiments',
                    'heterogeneous_distribution' : 0
                }
    
    experiment_folder = constant_parameters['experiment_folder']
    already_done = get_completed_experiments(experiment_folder)
    
    combinations = []
    
    keys = list(variable_parameters)
    for values in itertools.product(*map(variable_parameters.get, keys)):
        combinations.append(dict(zip(keys, values)))
    

    n_experiments = len(combinations)

    experiments = []

    for idx, parameters in enumerate(combinations) : 
        all_parameters = parameters | constant_parameters
        all_parameters['n_honest_workers'] = all_parameters['n_workers'] - all_parameters['n_byzantine_workers']
        infered_parameters = {}

        if (parameters['dataset_name'] == 'Purchase100') :
            n_classes = 100 
        elif (parameters['dataset_name'] == 'EMNIST') :
            n_classes = 62
        else : 
            n_classes = 10 

        all_parameters['n_classes'] = n_classes

        if torch.cuda.is_available(): 
            n_gpu = gpu_list[gpu_selection % len(gpu_list)]
            device = torch.device(f"cuda:{n_gpu}")
            all_parameters['device'] = device
            
            gpu_selection += 1
        else : 
            all_parameters['device'] = 'cpu'

        attack_parameters = get_attack_parameters(all_parameters)
        infered_parameters['attack_parameters'] = attack_parameters

        aggregator_parameters = get_aggregator_parameters(all_parameters)
        infered_parameters['aggregator_parameters'] = aggregator_parameters

        pre_aggregator_parameters = get_pre_aggregator_parameters(all_parameters)
        infered_parameters['pre_aggregator_parameters'] = pre_aggregator_parameters

        criterion_parameters = get_criterion_parameters(all_parameters)
        infered_parameters['criterion_parameters'] = criterion_parameters

        experiment_id = get_id(all_parameters)
        infered_parameters['experiment_id'] = experiment_id
        infered_parameters['n_experiments'] = n_experiments
        
        if all_parameters['dataset_name'] == 'MNIST' or all_parameters['dataset_name'] == 'EMNIST' or all_parameters['dataset_name'] == 'Fashion_MNIST' or all_parameters['dataset_name'] == 'KMNIST' :
            infered_parameters['n_step'] =  5
            infered_parameters['lr'] = lr_MNIST
        elif all_parameters['dataset_name'] == 'EuroSAT' or all_parameters['dataset_name'] == 'STL10'  : 
            infered_parameters['n_step'] =  5
            infered_parameters['lr'] = lr_EuroSAT
        else:
            infered_parameters['n_step'] =  8
            infered_parameters['lr'] = lr_CIFAR10_Purchase100

        if experiment_id not in already_done : 
            kwargs = all_parameters | infered_parameters
            experiments.append(kwargs)

    print("NB EXP :", len(experiments))
    how_many_in_parallel = len(gpu_list)*2 if len(gpu_list) > 0 else 4
    mini_batch_of_combinations = split_list(experiments, how_many_in_parallel)

    try : 
        torch.multiprocessing.set_start_method('fork')
    
    except : 
        pass 

    for combination_batch in mini_batch_of_combinations:
        pool = Pool()
        pool.map(run, combination_batch)
        pool.close()
        pool.join()
        torch.cuda.empty_cache()
        gc.collect()

In [7]:
variable_parameters = {
                    'attack_name': ['ALIE', 'FOE'],
                    'aggregator_name': ['CwTM'],
                    'pre_aggregator_name': ['None'],
                    'criterion_name': ["NorthStar"],
                    'dataset_name': ['MNIST', 'Fashion_MNIST'],
                    'n_byzantine_workers' : [2], 
                    'alpha' : [10]
                }  
multiple_exp_jupyter(variable_parameters)


NB EXP : 4
ExperimentExperimentExperimentExperiment    FOE_CwTM_None_NorthStar_MNIST_2_10_5_8_0.9_ALIE_CwTM_None_NorthStar_MNIST_2_10_5_8_0.9_ALIE_CwTM_None_NorthStar_Fashion_MNIST_2_10_5_8_0.9_FOE_CwTM_None_NorthStar_Fashion_MNIST_2_10_5_8_0.9_    // / /4 4  4 4starts. starts. 
starts.

starts.
distribution dirichlet per clientdistribution dirichlet per client

distribution dirichlet per clientdistribution dirichlet per client

Training FOE_CwTM_None_NorthStar_Fashion_MNIST_2_10_5_8_0.9_ / 4  starts.
Training TrainingALIE_CwTM_None_NorthStar_MNIST_2_10_5_8_0.9_  ALIE_CwTM_None_NorthStar_Fashion_MNIST_2_10_5_8_0.9_ // 4  4 starts.
  starts.
Training FOE_CwTM_None_NorthStar_MNIST_2_10_5_8_0.9_ / 4  starts.


Process ForkPoolWorker-5:
Process ForkPoolWorker-6:
Process ForkPoolWorker-1:
Process ForkPoolWorker-8:
Process ForkPoolWorker-7:
Process ForkPoolWorker-3:
Process ForkPoolWorker-4:
Process ForkPoolWorker-2:
Traceback (most recent call last):
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*

KeyboardInterrupt: 

  File "/home/sandie/PROJECTS/wola/models.py", line 66, in forward
    x = self.conv1(x)
        ^^^^^^^^^^^^^
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/site-packages/torch/nn/modules/conv.py", line 554, in forward
    return self._conv_forward(input, self.weight, self.bias)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/site-packages/torch/nn/functional.py", line 830, in _max_pool2d
    return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sandie/miniconda3/envs/test_env/lib/python3.12/site-packages/torch/nn/modules/conv.py", line 549, in _conv_f